# Transform data from managed tables and load into silver tier

In [0]:
%sql
-- setting the catalog
USE jarvis_training_catalog.silver;

## Process: Load -> Clean + Enrich -> Write to silver tier

### Card table (no issues)

In [0]:
card_df = spark.table('jarvis_training_catalog.bronze.cards')

card_df.count()

6146

In [0]:
# first, we'll check drop any null values anywhere, same with duplicates
card_df = card_df.dropna(how='any')
card_df = card_df.dropDuplicates()

card_df.count()

6146

In [0]:
# write to silver tier
card_df.write.mode('overwrite').saveAsTable('cards')

### User table (no issues)

In [0]:
user_df = spark.table('jarvis_training_catalog.bronze.users')

user_df.count()

2000

In [0]:
# first, we'll check drop any null values anywhere, same with duplicates
user_df = user_df.dropna(how='any')
user_df = user_df.dropDuplicates()

user_df.count()

2000

In [0]:
# write to silver tier
user_df.write.mode('overwrite').saveAsTable('users')

### Transactions table

In [0]:
# we loaded date column as date instead of datetime, so let's re-read from the csv and fix it
volume_path = "/Volumes/jarvis_training_catalog/bronze/data"

transaction_with_date = spark.read.csv(f"{volume_path}/transaction_data_cleaned.csv", header=True, inferSchema=True)

In [0]:
transaction_with_date.write.mode('overwrite').saveAsTable('jarvis_training_catalog.bronze.transactions_with_date')

It seems as if the errors column is causing us problems when trying to visualize the table (not excepting errors column leads to an exception). Let's try and perform some transformations to resolve the problem.

In [0]:
query = '''
SELECT t1.id, t1.date, t1.client_id, t1.card_id, t1.amount, t1.use_chip, t1.merchant_id, t1.merchant_city, t1.merchant_state, t1.zip, t1.mcc, t1.errors FROM
jarvis_training_catalog.bronze.transactions_with_date t1
JOIN 
jarvis_training_catalog.bronze.transactions t2
ON t1.id = t2.id;
'''

transaction_df_fixed = spark.sql(query)

In [0]:
from pyspark.sql.functions import col, round
from pyspark.sql.types import StringType

# the null values in the errors column are raising exceptions, so let's replace them with a filler "No issue" string
transaction_df = transaction_df_fixed.withColumn(
    "errors",
    col("errors").cast(StringType())
).na.fill({"errors": "No issue"}).withColumn('fraud_label')


display(transaction_df.head(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
21698897,2018-08-20T22:58:00.000Z,1435,2360,38.19,Chip Transaction,16915,Alpharetta,GA,30004.0,5300,No issue
21698898,2018-08-20T22:58:00.000Z,1763,4297,-494.0,Swipe Transaction,15426,Clute,TX,77531.0,3390,No issue
21698899,2018-08-20T22:59:00.000Z,1147,2042,26.38,Swipe Transaction,19756,New York,NY,10002.0,7832,No issue
21698900,2018-08-20T22:59:00.000Z,1266,2478,22.67,Chip Transaction,20778,Aberdeen,MD,21001.0,7832,No issue
21698901,2018-08-20T23:05:00.000Z,1384,3723,54.61,Chip Transaction,58507,Kingston,Jamaica,null,5812,No issue


In [0]:
transaction_df.write.mode('overwrite').saveAsTable('transactions_unenhanced')

Doesn't look like we have any null values/duplicates in the card/user tables, which is great! Now, we'll work on enriching the transaction table using the `.json` files that provide mcc codes and fraud labels:
- The `train_fraud_labels.json` file indicates whether users were the target of a fraud scheme
- The `mcc_labels.json` file provides some extra information about the type of businesses customers made purchases from

### MCC Codes JSON

In [0]:
# load json files from volume
from pyspark.sql.functions import explode, col

# Load JSON (this becomes ONE row with many columns)
mcc_raw_df = spark.read.option('multiline', True).json(f"{volume_path}/mcc_codes.json")
mcc_raw_df.printSchema()

root
 |-- 1711: string (nullable = true)
 |-- 3000: string (nullable = true)
 |-- 3001: string (nullable = true)
 |-- 3005: string (nullable = true)
 |-- 3006: string (nullable = true)
 |-- 3007: string (nullable = true)
 |-- 3008: string (nullable = true)
 |-- 3009: string (nullable = true)
 |-- 3058: string (nullable = true)
 |-- 3066: string (nullable = true)
 |-- 3075: string (nullable = true)
 |-- 3132: string (nullable = true)
 |-- 3144: string (nullable = true)
 |-- 3174: string (nullable = true)
 |-- 3256: string (nullable = true)
 |-- 3260: string (nullable = true)
 |-- 3359: string (nullable = true)
 |-- 3387: string (nullable = true)
 |-- 3389: string (nullable = true)
 |-- 3390: string (nullable = true)
 |-- 3393: string (nullable = true)
 |-- 3395: string (nullable = true)
 |-- 3405: string (nullable = true)
 |-- 3504: string (nullable = true)
 |-- 3509: string (nullable = true)
 |-- 3596: string (nullable = true)
 |-- 3640: string (nullable = true)
 |-- 3684: string (null

In [0]:
display(mcc_raw_df)

1711,3000,3001,3005,3006,3007,3008,3009,3058,3066,3075,3132,3144,3174,3256,3260,3359,3387,3389,3390,3393,3395,3405,3504,3509,3596,3640,3684,3722,3730,3771,3775,3780,4111,4112,4121,4131,4214,4411,4511,4722,4784,4814,4829,4899,4900,5045,5094,5192,5193,5211,5251,5261,5300,5310,5311,5411,5499,5533,5541,5621,5651,5655,5661,5712,5719,5722,5732,5733,5812,5813,5814,5815,5816,5912,5921,5932,5941,5942,5947,5970,5977,6300,7011,7210,7230,7276,7349,7393,7531,7538,7542,7549,7801,7802,7832,7922,7995,7996,8011,8021,8041,8043,8049,8062,8099,8111,8931,9402
"Heating, Plumbing, Air Conditioning Contractors",Steelworks,Steel Products Manufacturing,Miscellaneous Metal Fabrication,Miscellaneous Fabricated Metal Products,Coated and Laminated Products,Steel Drums and Barrels,Fabricated Structural Metal Products,"Tools, Parts, Supplies Manufacturing",Miscellaneous Metals,"Bolt, Nut, Screw, Rivet Manufacturing",Leather Goods,Floor Covering Stores,Upholstery and Drapery Stores,"Brick, Stone, and Related Materials",Pottery and Ceramics,Non-Ferrous Metal Foundries,"Electroplating, Plating, Polishing Services",Non-Precious Metal Services,Miscellaneous Metalwork,Heat Treating Metal Services,Welding Repair,Ironwork,Gardening Supplies,Industrial Equipment and Supplies,Miscellaneous Machinery and Parts Manufacturing,"Lighting, Fixtures, Electrical Supplies",Semiconductors and Related Devices,Passenger Railways,Ship Chandlers,Railroad Passenger Transport,Railroad Freight,Computer Network Services,Local and Suburban Commuter Transportation,Passenger Railways,Taxicabs and Limousines,Bus Lines,Motor Freight Carriers and Trucking,Cruise Lines,Airlines,Travel Agencies,Tolls and Bridge Fees,Telecommunication Services,Money Transfer,"Cable, Satellite, and Other Pay Television Services","Utilities - Electric, Gas, Water, Sanitary","Computers, Computer Peripheral Equipment",Precious Stones and Metals,"Books, Periodicals, Newspapers","Florists Supplies, Nursery Stock and Flowers",Lumber and Building Materials,Hardware Stores,Lawn and Garden Supply Stores,Wholesale Clubs,Discount Stores,Department Stores,"Grocery Stores, Supermarkets",Miscellaneous Food Stores,Automotive Parts and Accessories Stores,Service Stations,Women's Ready-To-Wear Stores,Family Clothing Stores,"Sports Apparel, Riding Apparel Stores",Shoe Stores,"Furniture, Home Furnishings, and Equipment Stores",Miscellaneous Home Furnishing Stores,Household Appliance Stores,Electronics Stores,Music Stores - Musical Instruments,Eating Places and Restaurants,Drinking Places (Alcoholic Beverages),Fast Food Restaurants,"Digital Goods - Media, Books, Apps",Digital Goods - Games,Drug Stores and Pharmacies,"Package Stores, Beer, Wine, Liquor",Antique Shops,Sporting Goods Stores,Book Stores,"Gift, Card, Novelty Stores","Artist Supply Stores, Craft Shops",Cosmetic Stores,"Insurance Sales, Underwriting","Lodging - Hotels, Motels, Resorts",Laundry Services,Beauty and Barber Shops,Tax Preparation Services,Cleaning and Maintenance Services,"Detective Agencies, Security Services",Automotive Body Repair Shops,Automotive Service Shops,Car Washes,Towing Services,"Athletic Fields, Commercial Sports","Recreational Sports, Clubs",Motion Picture Theaters,Theatrical Producers,"Betting (including Lottery Tickets, Casinos)","Amusement Parks, Carnivals, Circuses","Doctors, Physicians",Dentists and Orthodontists,Chiropractors,"Optometrists, Optical Goods and Eyeglasses",Podiatrists,Hospitals,Medical Services,Legal Services and Attorneys,"Accounting, Auditing, and Bookkeeping Services",Postal Services - Government Only


This gives us one row with ~100 columns, instead of ~100 rows with the mcc information in the only additional column. Let's pivot the dataframe accordingly.

In [0]:
from pyspark.sql.functions import col
# get the column names as strings (all of the mcc codes)
codes = mcc_raw_df.columns

"""
- stack will convert rows to columns
- we tell it how many rows to create (len(codes)) and key value pairs `c` will evaluate the description of the mcc code business
- e.g. `1434` means the description for mcc code 1434
"""
mcc_df = (
    mcc_raw_df
    .selectExpr(
        "stack({}, {}) as (mcc, mcc_description)".format(
            len(codes),
            ", ".join([f"'{c}', `{c}`" for c in codes])
        )
    )
    .withColumn("mcc", col("mcc").cast("int"))
)

display(mcc_df.head(5))

mcc,mcc_description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products


In [0]:
# now write to a silver table
mcc_df.write.mode('overwrite').saveAsTable('mcc_codes')

### Fraud JSON

In [0]:
from pyspark.sql.functions import explode
from pyspark.sql.types import StructType, StructField, MapType, StringType

fraud_schema = StructType([
    StructField("target", MapType(StringType(), StringType()), nullable=True)
])
# specify a schema so that spark knows how to parse the json
# target is a field, each user within target contains user_id mapping to a string (Yes/No)

fraud_raw_df = spark.read.option('multiline', True).schema(fraud_schema).json(f"{volume_path}/train_fraud_labels.json")

In [0]:
# explode the dataframe to create a row for every user (currently we have all 2000 users in one row)
fraud_labels_df = (
    fraud_raw_df
    .select(explode("target").alias("key", "value"))
    .withColumnRenamed("key", "transaction_id")
    .withColumnRenamed("value", "fraud_label")
    .withColumn("transaction_id", col("transaction_id").cast("int"))
)
# fraud_raw_df.printSchema()
display(fraud_labels_df.head(10))

transaction_id,fraud_label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
fraud_labels_df.write.mode('overwrite').saveAsTable('fraud_labels')

Now we can enrich the transactions table by joining it with the `mcc_codes` \& `fraud_labels` tables

In [0]:
query = '''
SELECT * 
EXCEPT(M.mcc)
FROM
fraud_labels F 
JOIN
transactions_unenhanced T
ON F.transaction_id = T.id
JOIN 
mcc_codes M 
ON T.mcc = M.mcc
'''

enhanced_transactions_df = spark.sql(query)
display(enhanced_transactions_df.head(10))
enhanced_transactions_df.write.mode('overwrite').saveAsTable('transactions_enhanced')

transaction_id,fraud_label,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_description
19642844,No,19642844,2017-06-11T12:41:00.000Z,955,5723,20.05,Chip Transaction,83229,Wellsburg,WV,26070.0,5411,No issue,"Grocery Stores, Supermarkets"
19643014,No,19643014,2017-06-11T13:11:00.000Z,595,4666,22.42,Chip Transaction,65664,Cullman,AL,35055.0,5813,No issue,Drinking Places (Alcoholic Beverages)
19643187,No,19643187,2017-06-11T13:43:00.000Z,764,3727,64.21,Chip Transaction,10622,Warner Robins,GA,31088.0,5912,No issue,Drug Stores and Pharmacies
19644259,No,19644259,2017-06-11T17:37:00.000Z,852,1282,27.78,Chip Transaction,60569,Keystone Heights,FL,32656.0,5300,No issue,Wholesale Clubs
19645290,No,19645290,2017-06-12T00:44:00.000Z,1963,3317,49.58,Chip Transaction,88646,Vacaville,CA,95688.0,5812,No issue,Eating Places and Restaurants
19646018,No,19646018,2017-06-12T08:12:00.000Z,1098,5179,-97.0,Chip Transaction,50867,Hawarden,IA,51023.0,5541,No issue,Service Stations
19646476,No,19646476,2017-06-12T09:54:00.000Z,1237,2017,22.23,Online Transaction,96246,ONLINE,null,null,4784,No issue,Tolls and Bridge Fees
19647110,No,19647110,2017-06-12T12:07:00.000Z,1077,214,2.08,Chip Transaction,47033,Ukiah,CA,95482.0,4121,No issue,Taxicabs and Limousines
19647663,No,19647663,2017-06-12T13:43:00.000Z,1332,2406,-99.0,Swipe Transaction,59935,Port Chester,NY,10573.0,5499,No issue,Miscellaneous Food Stores
19648104,No,19648104,2017-06-12T15:15:00.000Z,1412,3067,522.16,Online Transaction,52073,ONLINE,null,null,4722,No issue,Travel Agencies
